In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import ipaddress
from datetime import datetime
import json
import os
from sklearn.metrics import classification_report


In [ ]:
df = pd.read_csv('Signature_logs.csv', parse_dates = ["timestamp"])

In [ ]:
df.shape

(25000, 20)

In [ ]:
df.isnull().sum()

timestamp            0
host                 0
msg_id               0
severity             0
conn_id              0
ingress_iface        0
egress_iface         0
ingress_zone         0
egress_zone          0
src_ip               0
dst_ip               0
protocol             0
src_port             0
dst_port             0
acl_name             0
user                 0
action               0
is_anomaly           0
anomaly_type     24105
message              0
dtype: int64

In [ ]:
expected_cols = [
    "timestamp","host","msg_id","severity","message","src_ip","src_port","dst_ip","dst_port",
    "protocol","ingress_iface","egress_iface","ingress_zone","egress_zone","acl_name","user",
    "action","conn_id","is_anomaly","anomaly_type"
]
missing = set(expected_cols) - set(df.columns)
if missing:
    raise ValueError(f"Dataset missing required columns: {missing}")


In [ ]:
df.shape

(25000, 25)

In [ ]:
df = df.sort_values("timestamp")
df["protocol"] = df["protocol"].str.lower().fillna("tcp")
df["src_port"] = pd.to_numeric(df["src_port"], errors="coerce")
df["dst_port"] = pd.to_numeric(df["dst_port"], errors="coerce")

In [ ]:


def ip_to_int(ip):
    try: return int(ipaddress.ip_address(ip))
    except: return np.nan

def proto_to_num(p):
    p = str(p).lower()
    return 6 if p=="tcp" else 17 if p=="udp" else -1

# compact features
df["msg_id_code"] = df["msg_id"].astype("category").cat.codes
df["protocol_num"] = df["protocol"].apply(proto_to_num)
df["hour"] = df["timestamp"].dt.hour
df["minute"] = df["timestamp"].dt.minute
df["weekday"] = df["timestamp"].dt.weekday

feat_cols = ["msg_id_code","severity","src_port","dst_port","protocol_num","hour","minute","weekday"]
X = df[feat_cols].astype(float).values
y = df["is_anomaly"].astype(int).values


In [ ]:
df


,timestamp,host,msg_id,severity,conn_id,ingress_iface,egress_iface,ingress_zone,egress_zone,src_ip,...,user,action,is_anomaly,anomaly_type,message,msg_id_code,protocol_num,hour,minute,weekday
0,2025-08-10 12:20:03.305926,fw2.site,FTD-6-430003,6,39576,outside,dmz,Inside,DC,52.91.51.198,...,eve,allow_or_block,False,NaN,"%FTD-6-430003: AccessControlRuleAction: Allow,...",15,6,12,20,6
1,2025-08-10 12:20:23.305926,fw1.site,ASA-6-302016,6,43430,inside,inside,DC,Branch,9.22.49.27,...,unknown,deny,False,NaN,%ASA-6-302016: Denied TCP due to embryonic lim...,9,17,12,20,6
2,2025-08-10 12:20:25.305926,fw2.site,ASA-6-106100,6,58121,dmz,outside,Outside,Outside,251.199.64.191,...,alice,count,False,NaN,%ASA-6-106100: access-list Deny_Suspicious COU...,3,6,12,20,6
3,2025-08-10 12:20:31.305926,fw2.site,FTD-6-430003,6,79589,dmz,outside,Branch,Inside,188.248.148.87,...,unknown,allow_or_block,False,NaN,"%FTD-6-430003: AccessControlRuleAction: Allow,...",15,17,12,20,6
4,2025-08-10 12:20:57.305926,fw2.site,FTD-6-430002,6,87424,dmz,inside,Branch,Inside,24.173.249.59,...,unknown,observe,False,NaN,"%FTD-6-430002: Connection Start, SrcIP: 24.173...",14,6,12,20,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,2025-08-12 12:18:52.305926,fw1.site,ASA-6-302013,6,36972,inside,outside,Outside,Outside,84.241.131.168,...,unknown,allow,False,NaN,%ASA-6-302013: Built TCP connection 36972 for ...,6,6,12,18,1
24996,2025-08-12 12:19:06.305926,fw2.site,FTD-6-302015,6,81238,outside,outside,DC,Inside,126.172.192.223,...,dave,deny,False,NaN,%FTD-6-302015: Denied TCP due to ACL 81238 DC-...,12,17,12,19,1
24997,2025-08-12 12:19:12.305926,fw2.site,ASA-6-302016,6,51493,dmz,inside,Inside,DC,181.133.11.5,...,bob,deny,False,NaN,%ASA-6-302016: Denied TCP due to embryonic lim...,9,17,12,19,1
24998,2025-08-12 12:19:39.305926,fw1.site,ASA-6-106100,6,46759,outside,inside,Inside,Branch,175.131.240.233,...,dave,count,False,NaN,%ASA-6-106100: access-list Default_Deny COUNT ...,3,17,12,19,1


In [ ]:
df['is_anomaly'].value_counts()

is_anomaly
False    24105
True       895
Name: count, dtype: int64

In [ ]:
len(df['msg_id_code'].unique())

16

In [ ]:
y

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
scaler = StandardScaler()
mask_norm = df["is_anomaly"] == False
X_norm = scaler.fit_transform(X[mask_norm])

iso = IsolationForest(
    n_estimators=200, max_samples=0.8, contamination=0.03,
    random_state=42, n_jobs=1
).fit(X_norm)

scores = iso.decision_function(scaler.transform(X))
preds  = iso.predict(scaler.transform(X))  # -1 anomaly, 1 normal

df["iforest_score"] = scores
df["iforest_pred"]  = preds

In [ ]:
print(classification_report(df["is_anomaly"], (df["iforest_pred"]==-1).astype(int)))
top_iforest = df.sort_values("iforest_score").head(50)
top_iforest[["timestamp","msg_id","src_ip","dst_ip","src_port","dst_port","anomaly_type","iforest_score"]].head(10)

              precision    recall  f1-score   support

       False       0.96      0.97      0.97     24105
        True       0.06      0.05      0.06       895

    accuracy                           0.94     25000
   macro avg       0.51      0.51      0.51     25000
weighted avg       0.93      0.94      0.93     25000



,timestamp,msg_id,src_ip,dst_ip,src_port,dst_port,anomaly_type,iforest_score
2462,2025-08-10 17:00:36.305926,ASA-4-106023,154.3.188.103,89.48.108.19,61005,61863,NaN,-0.100677
1561,2025-08-10 15:16:00.305926,ASA-4-106023,217.237.230.122,244.68.201.110,65467,59405,NaN,-0.088680
17554,2025-08-11 21:58:56.305926,ASA-4-106023,22.226.141.78,16.10.208.221,62735,55547,NaN,-0.081629
6254,2025-08-11 00:14:06.305926,ASA-4-106023,227.5.70.69,195.41.123.203,61745,65199,NaN,-0.079543
16509,2025-08-11 19:55:28.305926,ASA-4-106023,104.55.204.68,153.193.167.248,54419,65213,NaN,-0.079435
5087,2025-08-10 21:59:57.305926,ASA-4-106023,249.23.234.155,45.133.224.24,50726,38522,NaN,-0.078161
5104,2025-08-10 22:01:56.305926,ASA-4-106023,82.18.110.125,13.25.142.74,25051,54550,NaN,-0.076898
18096,2025-08-11 22:57:49.305926,ASA-4-106023,218.184.10.84,117.111.131.106,65478,27306,NaN,-0.073167
16553,2025-08-11 20:01:04.305926,ASA-4-106023,14.90.134.192,185.21.182.236,51723,60871,NaN,-0.071113
2928,2025-08-10 17:56:17.305926,ASA-4-106023,1.124.221.150,188.144.237.200,58608,44860,NaN,-0.069871


In [ ]:
g = (df.assign(minute_bucket=df["timestamp"].dt.floor("T"))
       .groupby(["minute_bucket","msg_id"])
       .size().rename("count").reset_index()
       .sort_values(["msg_id","minute_bucket"]))

/var/folders/n4/5dp991c57pxg_94804zmxngw0000gn/T/ipykernel_39109/2935899823.py:1: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  g = (df.assign(minute_bucket=df["timestamp"].dt.floor("T"))


In [ ]:
def add_z(d):
    c = d["count"].astype(float)
    mu = c.rolling(120, min_periods=30).mean()
    sd = c.rolling(120, min_periods=30).std(ddof=0).replace(0, np.nan)
    z  = ((c - mu) / sd).fillna(0.0)
    out = d.copy(); out["zscore"] = z; return out

In [ ]:
g = g.groupby("msg_id", group_keys=False).apply(add_z)
spikes = g[g["zscore"] > 4.0].sort_values("zscore", ascending=False)
spikes.head(10)

/var/folders/n4/5dp991c57pxg_94804zmxngw0000gn/T/ipykernel_39109/1196669575.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  g = g.groupby("msg_id", group_keys=False).apply(add_z)


,minute_bucket,msg_id,count,zscore
17489,2025-08-12 07:52:00,FTD-6-302014,5,6.942723
4756,2025-08-11 00:07:00,FTD-6-302015,5,6.620511
16204,2025-08-12 04:38:00,FTD-6-430003,5,6.112457
1494,2025-08-10 15:59:00,ASA-6-302014,5,6.057131
8750,2025-08-11 10:07:00,ASA-5-111010,5,5.989221
13760,2025-08-11 22:39:00,ASA-6-302014,5,5.989221
6493,2025-08-11 04:29:00,ASA-6-106102,5,5.862025
9158,2025-08-11 11:10:00,ASA-6-106102,5,5.597179
7753,2025-08-11 07:35:00,ASA-6-302013,5,5.597179
16583,2025-08-12 05:35:00,ASA-6-605005,5,5.565361


In [ ]:
df.to_csv("asa_ftd_scored.csv", index=False)
top_iforest.to_csv("asa_ftd_top50_iforest.csv", index=False)
spikes.to_csv("asa_ftd_spikes.csv", index=False)